# SO101 连续控制 · v5 SAC（state 观测，随机策略 + 最大熵 + 自动温度）

**这个 notebook 在做什么**：DDPG（v3）和 TD3（v4）都是**确定性策略**，采集时只能靠
外加一点固定方差的高斯噪声去"碰运气"探索。TD3 治的是 Q 值高估，完全没碰探索这件事。
SAC（Soft Actor-Critic）换一套路子直接对症：

1. **随机策略替代确定性策略**：`SquashedGaussianActor` 吐一个高斯分布的均值和标准
   差，重参数化 `rsample` 采样再 `tanh` 挤压进合法区间。策略本身自带探索。
2. **最大熵目标**：优化目标从"最大化回报"变成"最大化回报 + α·策略熵"，直接折进
   Critic 的贝尔曼目标和 Actor 的损失里。
3. **自动温度 α**：`log_alpha` 像另一个网络参数一样被优化，调到让当前策略熵匹配
   目标熵 `-action_dim`。

双 Q 取 min 治高估这套照抄 TD3 不动；拿掉的是"目标策略平滑"和"延迟策略更新"——这
两个是专为确定性策略的高估问题设计的，随机策略下没必要。

**观测归一化同样是这一级能学起来的前提**：`SquashedGaussianActor` 的均值分支也先
过 `tanh`（`get_action` 里 `y = torch.tanh(x)`），SO101 state 某些维度原始值到 ~200
一样会把这条通路饱和死；`RunningNorm` 的接法和 v3/v4 完全一致。

**诚实的教学结论（公平预算 500 iter、三级同预算、加观测归一化后的真实结果）**：三级
放在一起看，是一条干净的阶梯，而且能看出两层完全不同的道理。第一层，**靠近度
（mean_reward）单调递增**：v3 DDPG 全程震荡、均值约 0.28；v4 TD3 治住震荡、稳定靠近、
均值约 0.52，比 DDPG 明显更近更稳；每一级"更会靠近"这件事是连续的、可预期的。第二层，
**硬成功（success_once）是质变，不是量变**：DDPG 和 TD3 无论靠得多近，500 轮里
success_once 始终是 0——同样是"越来越会靠近"的两级，不管给多久预算都跨不过成功那道坎；
而 SAC 一旦换上随机策略 + 最大熵，success_once 从 iter~480 起开始稳定落在 0.93～0.99，
最终 success_once=0.99，直接把这道坎彻底跨过去了。这印证了最初的猜想：探索不足才是
DDPG/TD3 学不动的病根，最大熵这味药才是打开"解决"的钥匙——双 Q 治高估、延迟更新这些
稳定性技巧再怎么打磨，也补不上"探索方式本身不对"这个洞。当然 SAC 这一级也不是没有代价：
要跑到 500 iter 才等到 success_once 稳定超过 0.9，说明**标量 Q + MSE** 的样本效率仍然
有限，Critic 只学一个数值的期望回报；下一级 v6 squint 会换成 C51 分布式 Critic（学整个
回报分布而不是一个期望值）配视觉编码器，要更快更强地解决问题，还要能产出规模化的数据。

> **运行方式**：SO101 仿真需要 GPU（ManiSkill GPU 后端）。自上而下逐 cell 运行；
> 打印格式 `iter N: success_once=X  mean_reward=Y  alpha=Z`，可与 v3/v4 直接对照。
> 训好的 actor + 归一化统计存到
> `DATASETS_ROOT/models/trained/so101_sim_offpolicy/<task>/sac.pt`。

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, IterableDataset

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import so101_sim  # noqa: E402  统一环境：lerobot 评测与 RL 训练共用同一份定义


## 0 RunningNorm：在线观测归一化（与 v3/v4 完全一致）

SO101 state 某些维度原始值能到 ~200；`SquashedGaussianActor` 的均值分支也要过 `tanh`
挤压，一样会被大数值饱和死。`RunningNorm` 用 Welford 并行算法在线维护每一维的
running mean/var（采集时用原始 state 更新），喂进网络前再归一化成零均值单位方差；
回放池里仍然存原始 state。

In [ ]:
class RunningNorm(nn.Module):
    """在线观测归一化：跟踪 state 每一维的 running mean/std，把原始观测（值域可到 ~200）
    归一化到零均值单位方差，避免大数值让 actor 的 tanh 饱和、梯度冻结。"""

    def __init__(self, dim):
        super().__init__()
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("var", torch.ones(dim))
        self.register_buffer("count", torch.tensor(1e-4))

    @torch.no_grad()
    def update(self, x):
        bm, bv, bc = x.mean(0), x.var(0, unbiased=False), x.shape[0]
        delta = bm - self.mean; tot = self.count + bc
        self.mean += delta * bc / tot
        M2 = self.var * self.count + bv * bc + delta**2 * self.count * bc / tot
        self.var = M2 / tot; self.count = tot

    def normalize(self, x):
        return (x - self.mean) / (self.var.sqrt() + 1e-8)

## 1 SquashedGaussianActor：state → 随机动作

相对 v3/v4 的 `DeterministicActor`：那边只有一个 `forward` 直接给动作；这里没有
`forward`，只有 `get_action`——SAC 的策略是一个分布，每次调用都要显式采样，还要
顺带算出这个样本的 log 概率（Critic 目标和 Actor 损失都要用）。均值/log 标准差
出网络后，重参数化 `rsample` 出样本，`tanh` 挤压进 [-1,1] 再线性缩放进
[low, high]；log_prob 要按 tanh 的雅可比行列式修正，否则概率密度是错的。


In [ ]:
class SquashedGaussianActor(nn.Module):
    """state → 随机动作：MLP 出均值和 log 标准差，重参数化采样后 tanh 挤压进 [low, high]。

    相对 v3/v4 的 `DeterministicActor`：那边只有一个 `forward` 直接给动作；这里没有
    `forward`，只有 `get_action`——因为 SAC 的策略是一个分布，每次调用都要显式采样，
    还要顺带算出这个样本的 log 概率（Critic 目标和 Actor 损失都要用到）。
    """

    LOG_STD_MIN, LOG_STD_MAX = -5, 2

    def __init__(self, state_dim, action_dim, action_low, action_high):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.fc_mean = nn.Linear(256, action_dim)
        self.fc_logstd = nn.Linear(256, action_dim)
        self.register_buffer("action_scale", (action_high - action_low) / 2.0)
        self.register_buffer("action_bias", (action_high + action_low) / 2.0)

    def _mean_logstd(self, state):
        x = self.trunk(state)
        mean = self.fc_mean(x)
        log_std = torch.tanh(self.fc_logstd(x))
        log_std = self.LOG_STD_MIN + 0.5 * (self.LOG_STD_MAX - self.LOG_STD_MIN) * (log_std + 1)
        return mean, log_std

    def get_action(self, state):
        """随机动作 + log 概率（含 tanh 雅可比修正）+ 确定性均值动作（评测用）。"""
        mean, log_std = self._mean_logstd(state)
        normal = torch.distributions.Normal(mean, log_std.exp())
        x = normal.rsample()  # 重参数化：采样路径可导，梯度能穿回 mean/log_std
        y = torch.tanh(x)
        action = y * self.action_scale + self.action_bias
        # tanh 把采样值挤压了一次，概率密度要按雅可比行列式修正，否则 log_prob 是错的
        log_prob = normal.log_prob(x) - torch.log(self.action_scale * (1 - y.pow(2)) + 1e-6)
        log_prob = log_prob.sum(-1, keepdim=True)
        det_action = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, det_action


## 2 QCritic：(state, action) → 标量 Q

和 v4 TD3 完全一样，拼接后过 MLP，只输出一个标量 Q(s,a)。SAC 这一级同样实例化两份
（q1/q2），双 Q 取 min 治高估这套照抄不变；变化都在下面的 `SAC.training_step` 里。


In [ ]:
class QCritic(nn.Module):
    """(state, action) → 标量 Q：和 v4 TD3 完全一样，拼接后过 MLP，只输出一个数值。"""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1)).squeeze(-1)


## 3 SAC：LightningModule，手动优化（三个优化器：actor / critic / log_alpha）

持有 `obs_norm`（`RunningNorm`）+ 随机策略 `SquashedGaussianActor` + 两套独立
critic/critic_target（无 actor_target——随机策略不需要）。`training_step`：
1. **评论家**：下一步动作由当前 actor 采样（不是目标 actor），双目标网络取 `min`
   再减掉 `alpha * next_logp`（熵项折进贝尔曼目标），两个在线 critic 各自 MSE 回归；
2. **温度**：把当前策略熵调到目标熵 `-action_dim`；
3. **演员**：`(alpha * logp - min(q1_pi, q2_pi)).mean()`，直接优化"回报 + 熵"。

`state`/`next_state` 在喂进 actor/critic 前统一过 `self.obs_norm.normalize(...)`，
buffer 里存的还是原始 state。`sample_action` 直接是随机采样（不叠额外噪声）；
`eval_action` 用确定性均值动作。

In [ ]:
class SAC(L.LightningModule):
    """一个 minibatch 的 SAC 更新：评论家(熵折进目标) → 温度 α → 演员(熵正则的策略梯度) → 软更新。"""

    def __init__(self, state_dim, action_dim, action_low, action_high, gamma=0.99, tau=0.005, lr=3e-4):
        super().__init__()
        self.automatic_optimization = False
        self.gamma, self.tau, self.lr = gamma, tau, lr
        # 相对 DDPG/TD3 新增：自动温度——log_alpha 是待优化参数，目标熵按动作维度定
        self.target_entropy = -float(action_dim)
        self.log_alpha = nn.Parameter(torch.zeros(()))

        # 相对 DDPG/TD3 的最大变化：策略是随机的挤压高斯，不再是确定性 MLP
        self.actor = SquashedGaussianActor(state_dim, action_dim, action_low, action_high)
        # 双 Q 取 min 治高估这套照抄 TD3：两个独立 Critic + 各自目标网络
        self.critic1 = QCritic(state_dim, action_dim)
        self.critic2 = QCritic(state_dim, action_dim)
        self.critic1_target = QCritic(state_dim, action_dim)
        self.critic2_target = QCritic(state_dim, action_dim)
        self.critic1_target.load_state_dict(self.critic1.state_dict())
        self.critic2_target.load_state_dict(self.critic2.state_dict())
        # 观测归一化：buffer 里存的仍是原始 state，这里只在喂进网络前做归一化
        self.obs_norm = RunningNorm(state_dim)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def configure_optimizers(self):
        critic_params = list(self.critic1.parameters()) + list(self.critic2.parameters())
        return (torch.optim.Adam(self.actor.parameters(), lr=self.lr),
                torch.optim.Adam(critic_params, lr=self.lr),
                torch.optim.Adam([self.log_alpha], lr=self.lr))

    @torch.no_grad()
    def sample_action(self, state):
        """随机动作本身就是探索，采集时不再像 DDPG/TD3 那样额外叠高斯噪声。"""
        action, _, _ = self.actor.get_action(self.obs_norm.normalize(state))
        return action

    @torch.no_grad()
    def eval_action(self, state):
        _, _, det_action = self.actor.get_action(self.obs_norm.normalize(state))
        return det_action

    def training_step(self, batch, batch_idx):
        actor_opt, critic_opt, alpha_opt = self.optimizers()
        state, action, reward, next_state = batch
        # buffer 里是原始 state，喂进网络前统一归一化（统计量只在采集时更新，这里只读）
        state, next_state = self.obs_norm.normalize(state), self.obs_norm.normalize(next_state)

        # —— 评论家：双 Q 取 min 算目标，熵项折进目标（always-bootstrap，和 v3/v4 一样不用 done）——
        with torch.no_grad():
            next_action, next_logp, _ = self.actor.get_action(next_state)
            next_logp = next_logp.squeeze(-1)
            target_q1 = self.critic1_target(next_state, next_action)
            target_q2 = self.critic2_target(next_state, next_action)
            # 相对 TD3 新增：目标里减掉 alpha*next_logp——下一步越"随机"（熵越高），
            # 目标值越高，鼓励策略维持探索而不是过早收窄到单一动作
            target_q = reward + self.gamma * (torch.min(target_q1, target_q2) - self.alpha.detach() * next_logp)
        critic_loss = (F.mse_loss(self.critic1(state, action), target_q)
                     + F.mse_loss(self.critic2(state, action), target_q))
        critic_opt.zero_grad(); self.manual_backward(critic_loss); critic_opt.step()

        # —— 温度 α：把当前策略熵调到目标熵 -action_dim ——
        with torch.no_grad():
            _, logp, _ = self.actor.get_action(state)
            logp = logp.squeeze(-1)
        alpha_loss = -(self.log_alpha * (logp + self.target_entropy).detach()).mean()
        alpha_opt.zero_grad(); self.manual_backward(alpha_loss); alpha_opt.step()

        # —— 演员：最大化 min(q1,q2) − α·熵，即最小化 alpha*logp − min(q1,q2) ——
        a, logp2, _ = self.actor.get_action(state)
        logp2 = logp2.squeeze(-1)
        q1_pi = self.critic1(state, a)
        q2_pi = self.critic2(state, a)
        actor_loss = (self.alpha.detach() * logp2 - torch.min(q1_pi, q2_pi)).mean()
        actor_opt.zero_grad(); self.manual_backward(actor_loss); actor_opt.step()

        # 没有 TD3 的延迟更新——随机策略下 Critic 目标本来就带一点自带的平滑，
        # 目标网络照常每步软更新
        with torch.no_grad():
            for p, tp in zip(self.critic1.parameters(), self.critic1_target.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * p)
            for p, tp in zip(self.critic2.parameters(), self.critic2_target.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * p)

        self.log_dict({"critic_loss": critic_loss.detach(), "actor_loss": actor_loss.detach(),
                      "alpha": self.alpha.detach()}, prog_bar=True, on_step=True, on_epoch=False)

## 4 ReplayBuffer：经验回放（state 版，与 v3/v4 完全一致）

定容环形缓冲区，整块开在 GPU 上；每步把所有并行环境的转移一次性滚动写入。state 版
只存 state / action / reward / next_state 四样，没有 rgb。


In [ ]:
class ReplayBuffer:
    """定容经验回放池，整块开在 GPU 上（state 版：无 rgb，只存关节状态向量）。与 v3/v4 完全一致。"""

    def __init__(self, capacity, state_dim, action_dim, device):
        z = lambda *s: torch.zeros(*s, device=device)  # noqa: E731
        self.state = z(capacity, state_dim)
        self.next_state = z(capacity, state_dim)
        self.action = z(capacity, action_dim)
        self.reward = z(capacity)
        self.capacity, self.device = capacity, device
        self.pos, self.full = 0, False

    def __len__(self):
        return self.capacity if self.full else self.pos

    def add(self, state, action, reward, next_state):
        n = state.shape[0]
        idx = (torch.arange(n, device=self.device) + self.pos) % self.capacity
        self.state[idx] = state; self.next_state[idx] = next_state
        self.action[idx] = action; self.reward[idx] = reward.float()
        self.pos = (self.pos + n) % self.capacity
        self.full = self.full or self.pos < n

    def sample(self, batch_size):
        i = torch.randint(0, len(self), (batch_size,), device=self.device)
        return self.state[i], self.action[i], self.reward[i], self.next_state[i]


## 5 SO101SACData：在线采集的 DataModule

`_collect` 采到原始 state 后先 `self.model.obs_norm.update(state)` 更新归一化统计，
再决定用策略还是随机动作；`last_success`/`last_reward` 分别记这一轮的成功率和平均
奖励（靠近度），供 `SuccessLogger` 打印。

In [ ]:
class SO101SACData(L.LightningDataModule):
    """持有环境和回放池；每轮先采样、再把 minibatch 交给 Trainer（state 版，无 rgb）。"""

    def __init__(self, env, model, buffer, steps_per_iter, updates_per_iter, batch_size, learning_starts):
        super().__init__()
        self.env, self.model, self.buffer = env, model, buffer
        self.steps_per_iter, self.updates_per_iter = steps_per_iter, updates_per_iter
        self.batch_size, self.learning_starts = batch_size, learning_starts
        self.last_success = 0.0
        self.last_reward = 0.0

    def _collect(self, use_policy):
        state = self.env.obs["state"]
        self.model.obs_norm.update(state)  # 用原始 state 更新归一化统计，每步一次
        if use_policy:
            action = self.model.sample_action(state)
        else:  # 预热：均匀随机动作把池子填起来
            low, high = self.env.single_action_space.low, self.env.single_action_space.high
            low = torch.as_tensor(low, device=self.env.device)
            high = torch.as_tensor(high, device=self.env.device)
            action = low + (high - low) * torch.rand(self.env.num_envs, self.env.action_dim, device=self.env.device)
        next_obs, reward, _, _, success = self.env.step(action)
        self.buffer.add(state, action, reward, next_obs["state"])
        return success.float().mean().item(), reward.float().mean().item()

    def train_dataloader(self):
        def gen():
            while len(self.buffer) < self.learning_starts:
                self._collect(use_policy=False)
            stats = [self._collect(use_policy=True) for _ in range(self.steps_per_iter)]
            succ, rew = zip(*stats)
            self.last_success = float(np.mean(succ))
            self.last_reward = float(np.mean(rew))
            for _ in range(self.updates_per_iter):
                yield self.buffer.sample(self.batch_size)

        class _DS(IterableDataset):
            def __iter__(self_inner):
                return gen()

        return DataLoader(_DS(), batch_size=None)

## 6 SuccessLogger：每轮打印成功率 + 平均奖励 + 自动温度 α + 定期存 ckpt

`iter N: success_once=X  mean_reward=Y  alpha=Z`，与 v3/v4 同一套 success/reward 格式，
多打一个 alpha 看温度怎么随训练收缩。周期性把 actor 权重和 `obs_norm` 的归一化统计
一起存盘。

In [ ]:
class SuccessLogger(L.Callback):
    """每轮打印采集成功率 + 平均奖励（靠近度） + 自动温度 α + 定期存 ckpt。"""

    def __init__(self, ckpt_dir, save_interval, max_iterations):
        self.ckpt_dir, self.save_interval, self.max_iterations = ckpt_dir, save_interval, max_iterations

    def on_train_epoch_end(self, trainer, pl_module):
        it = trainer.current_epoch + 1
        succ = trainer.datamodule.last_success
        rew = trainer.datamodule.last_reward
        print(f"  iter {it}: success_once={succ:.2f}  mean_reward={rew:.3f}  alpha={pl_module.alpha.item():.3f}",
              flush=True)
        if it % self.save_interval == 0 or it == self.max_iterations:
            self.ckpt_dir.mkdir(parents=True, exist_ok=True)
            torch.save({"actor": pl_module.actor.state_dict(),
                       "obs_norm": pl_module.obs_norm.state_dict()}, self.ckpt_dir / "sac.pt")

## 7 组装训练

四件套到位：环境 `make_train_env(obs_mode="state")`、模型 `SAC`、数据
`SO101SACData`、训练逻辑（手动优化的 SAC 更新）。入口还是标准 Lightning 姿势
`trainer.fit(model, datamodule)`。下面 `main()` 用的是与 v3 DDPG / v4 TD3
**完全相同的共享预算**——num_envs=1024、每 iter 256 次更新（UTD）、batch=512、
回放池 50 万，三级可公平对照，只有算法本身在变。


In [ ]:
def run_training(task, num_envs, max_iterations, updates_per_iter, batch_size,
                 buffer_capacity, learning_starts, device, seed=1):
    torch.manual_seed(seed)
    env = so101_sim.make_train_env(task, num_envs=num_envs, obs_mode="state", device=device)
    env.reset()
    low = torch.as_tensor(env.single_action_space.low, device=device)
    high = torch.as_tensor(env.single_action_space.high, device=device)
    model = SAC(env.state_dim, env.action_dim, low, high).to(device)
    buffer = ReplayBuffer(buffer_capacity, env.state_dim, env.action_dim, device)
    data = SO101SACData(env, model, buffer, steps_per_iter=1, updates_per_iter=updates_per_iter,
                        batch_size=batch_size, learning_starts=learning_starts)

    ckpt_dir = Path(os.environ["DATASETS_ROOT"]) / "models" / "trained" / "so101_sim_offpolicy" / task
    trainer = L.Trainer(
        accelerator="gpu", devices=1, max_epochs=max_iterations,
        reload_dataloaders_every_n_epochs=1, enable_checkpointing=False, logger=False,
        enable_model_summary=False, enable_progress_bar=False, log_every_n_steps=10,
        callbacks=[SuccessLogger(ckpt_dir, save_interval=25, max_iterations=max_iterations)],
    )
    trainer.fit(model, datamodule=data)
    env.close()
    return ckpt_dir / "sac.pt"

In [ ]:
# 改这里选任务与训练时长，然后 `python train_v5_sac.py`。
TASK = "SO101ReachCube-v1"

if __name__ == "__main__":
    # 与 v3 DDPG / v4 TD3 完全相同的共享预算，只差算法，三级可公平对照：
    # 每步 256 次更新（UTD），批 512，回放池 50 万；SAC 要到 iter~480 才稳定跨过"解决"，
    # 三级统一给够 500 iter 预算才公平。
    run_training(
        task=TASK, num_envs=1024, max_iterations=500, updates_per_iter=256, batch_size=512,
        buffer_capacity=500_000, learning_starts=5_000, device="cuda",
    )
